[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_12/TSA_ch12_business_cycle/TSA_ch12_business_cycle.ipynb)

# TSA_ch12_business_cycle

Business cycles in the frequency domain: quarterly growth and Hodrick-Prescott cycles of US real GDP (1947-2019) and Romanian real GDP (1995-2019); smoothed spectra and the share of the variance in the business-cycle band of 6-32 quarters.

*Time Series Analysis (TSA), Chapter 12: Spectral analysis. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_12'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # default size about 7 inches wide (as in MFM); a chart is shown 11-13 cm wide on a slide, so these font sizes
    # give 7-9 pt text there. Multi-panel charts should stay near 7-7.6 inches wide: at 10-11 inches the axis
    # text drops to about 5 pt on the slide.
    rc['figure.figsize'] = (7.0, 3.4)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)
    _remember_tight_layout()


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
from scipy import signal

from statsmodels.tsa.filters.hp_filter import hpfilter

SEED = 2026

RO_GDP = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')

GDP_END = '2019-12-31'

BC_BAND = (6, 32)

HP_LAMBDA = 1600

LOAD_FILE = 'ch4_ro_load_hourly.csv'

LOAD_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/Quantlets/Ch_04/ch4_ro_load_hourly.csv'

BW = 0.65

AR2 = (1.5, -0.75)

DFT_EXAMPLE = [4.0, 2.0, 0.0, 2.0]

HERE = "."

def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def periodogram(x, taper=False):
    """Periodogram I(nu_j) = |d(nu_j)|^2, d(nu_j) = T^(-1/2) sum_t x_t exp(-2 pi i nu_j t), at the Fourier
    frequencies nu_j = j / T, j = 1..[T/2] (mean removed). With taper=True a Hann (cosine bell) taper is applied
    and rescaled so that the total power is unchanged."""
    x = np.asarray(x, float)
    x = x - x.mean()
    T = len(x)
    if taper:
        w = 0.5 * (1 - np.cos(2 * np.pi * (np.arange(T) + 0.5) / T))
        x = x * w / np.sqrt(np.mean(w ** 2))
    I = np.abs(np.fft.fft(x)) ** 2 / T
    j = np.arange(1, T // 2 + 1)
    return j / T, I[j]


def daniell(I, m):
    """Daniell smoother: the average of the 2m + 1 periodogram ordinates around each frequency (reflected at the
    ends); approximately chi-square with df = 2(2m + 1) degrees of freedom."""
    if m == 0:
        return np.asarray(I, float)
    k = np.ones(2 * m + 1) / (2 * m + 1)
    return np.convolve(np.pad(np.asarray(I, float), m, mode='reflect'), k, mode='valid')


def chi2_band(fhat, df, level=0.95):
    """Approximate confidence interval for f(nu): [df fhat / chi2_df(1 - a/2), df fhat / chi2_df(a/2)]."""
    a = 1 - level
    return df * fhat / stats.chi2.ppf(1 - a / 2, df), df * fhat / stats.chi2.ppf(a / 2, df)


def welch(x, nperseg):
    """Welch estimate (Hann windows, 50% overlap), rescaled to the convention f(nu) = sum_h gamma(h) e^(-2 pi i nu h)."""
    nu, P = signal.welch(np.asarray(x, float) - np.mean(x), fs=1.0, window='hann', nperseg=nperseg)
    return nu[1:], P[1:] / 2


def arma_spectrum(nu, phi=(), theta=(), sigma2=1.0):
    """Spectral density of an ARMA(p,q): sigma^2 |theta(e^(-2 pi i nu))|^2 / |phi(e^(-2 pi i nu))|^2."""
    z = np.exp(-2j * np.pi * np.asarray(nu, float))
    num = np.ones_like(z) + sum(t * z ** (k + 1) for k, t in enumerate(theta))
    den = 1 - sum(p * z ** (k + 1) for k, p in enumerate(phi))
    return sigma2 * np.abs(num) ** 2 / np.abs(den) ** 2


def simulate_arma(phi, theta, T, seed=SEED, burn=500):
    rng = np.random.default_rng(seed)
    e = rng.standard_normal(T + burn)
    x = np.zeros(T + burn)
    for t in range(T + burn):
        x[t] = e[t] + sum(th * e[t - k - 1] for k, th in enumerate(theta) if t - k - 1 >= 0) \
            + sum(p * x[t - k - 1] for k, p in enumerate(phi) if t - k - 1 >= 0)
    return x[burn:]


def fisher_g(I):
    """Fisher's (1929) g test of a hidden periodicity: g = max I_j / sum I_j over j = 1..m (nu_j < 1/2), with the
    exact p-value P(g > x) = sum_k (-1)^(k-1) C(m, k) (1 - kx)^(m-1), k <= 1/x."""
    I = np.asarray(I, float)
    m = len(I)
    g = I.max() / I.sum()
    from math import comb
    p = sum((-1) ** (k - 1) * comb(m, k) * (1 - k * g) ** (m - 1) for k in range(1, int(1 / g) + 1))
    return {'g': float(g), 'm': int(m), 'p': float(min(max(p, 0.0), 1.0)), 'j': int(np.argmax(I) + 1)}


def gph_spec(x, m=None):
    """GPH (Geweke and Porter-Hudak 1983): OLS slope of log I(nu_j) on -log(4 sin^2(pi nu_j)), j = 1..m, is d;
    standard error pi / sqrt(24 m)."""
    nu, I = periodogram(x)
    m = m or int(np.floor(len(x) ** BW))
    X = -np.log(4 * np.sin(np.pi * nu[:m]) ** 2)
    b = np.polyfit(X, np.log(I[:m]), 1)
    return {'d': float(b[0]), 'c': float(b[1]), 'se': float(np.pi / np.sqrt(24 * m)), 'm': int(m),
            'nu_m': float(nu[m - 1])}


def morlet_cwt(x, periods, w0=6.0):
    """Continuous wavelet transform with a Morlet wavelet (Torrence and Compo 1998), computed by FFT;
    returns the wavelet power |W(s, t)|^2 / variance for the given Fourier periods (in observations)."""
    x = np.asarray(x, float)
    x = (x - x.mean()) / x.std()
    T = len(x)
    n = 1 << int(np.ceil(np.log2(2 * T)))
    xf = np.fft.fft(x, n)
    om = 2 * np.pi * np.fft.fftfreq(n)
    fourier_factor = 4 * np.pi / (w0 + np.sqrt(2 + w0 ** 2))
    P = np.zeros((len(periods), T))
    for i, per in enumerate(periods):
        s = per / fourier_factor
        psi = np.pi ** -0.25 * np.sqrt(2 * np.pi * s) * np.exp(-0.5 * (s * om - w0) ** 2) * (om > 0)
        P[i] = np.abs(np.fft.ifft(xf * psi)[:T]) ** 2
    return P


def sunspots():
    """Yearly mean sunspot number 1700-2008 (Wolf / SILSO series as shipped with statsmodels)."""
    s = load_statsmodels('sunspots')
    s.index = s.index.year
    return s


def gdp_growth(end=GDP_END):
    """Quarterly growth of real GDP, 100 x log difference: United States (FRED GDPC1) and Romania (Eurostat)."""
    us = read_fred('GDPC1').loc[:end]
    ro = read_eurostat(*RO_GDP).loc[:end]
    return {'US': (100 * np.log(us)).diff().dropna(), 'RO': (100 * np.log(ro)).diff().dropna(),
            'US_level': 100 * np.log(us), 'RO_level': 100 * np.log(ro)}


def hp_cycle(level, lam=HP_LAMBDA):
    cyc, trend = hpfilter(level, lamb=lam)
    return cyc


def load_hourly():
    """Hourly electricity load of Romania, MW (ENTSO-E extract of Chapter 4), on a regular UTC hourly grid."""
    s = None
    local = [os.path.join(HERE, '..', 'Ch_04', LOAD_FILE)] + [os.path.join(d, 'Quantlets', 'Ch_04', LOAD_FILE)
                                                                for d in ('.', '..', '../..', '../../..')]
    for src in [p for p in local if os.path.exists(p)] + [LOAD_RAW]:
        try:
            s = pd.read_csv(src, index_col=0, parse_dates=True).iloc[:, 0]
            break
        except Exception:
            continue
    s = s[~s.index.duplicated()].sort_index()
    s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq='h')).interpolate()
    return s.rename('load_MW')


def fig_gdp_series(save_it=True):
    """US and Romanian real GDP: quarterly growth and HP cycle (lambda = 1600), up to 2019 Q4."""
    G = gdp_growth()
    fig, axes = plt.subplots(1, 2, figsize=(9.32, 2.95))
    out = {}
    for ax, k, c in zip(axes, ('US', 'RO'), (st.MainBlue, st.IDAred)):
        cyc = hp_cycle(G[k + '_level'])
        ax.plot(G[k].index, G[k].values, color=c, lw=0.9, label='quarterly growth, %' if k == 'US' else '_')
        ax.plot(cyc.index, cyc.values, color=st.Forest, lw=1.5, label='HP cycle, % of trend' if k == 'US' else '_')
        ax.axhline(0, color=st.DarkText, lw=0.5)
        ax.set_title('United States' if k == 'US' else 'Romania')
        out[k] = {'T': int(len(G[k])), 'start': str(G[k].index[0].date()), 'end': str(G[k].index[-1].date()),
                  'mean': float(G[k].mean()), 'sd': float(G[k].std()), 'cyc_sd': float(cyc.std())}
    st.fig_legend_bottom(fig, ncol=2, y=0.03)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save('tsa_ch12_gdp_series', save_it)
    return out


def fig_gdp_spectra(save_it=True):
    """Smoothed spectra (Daniell, m = 2) of GDP growth and of the HP cycle, as a share of the variance, with the
    business-cycle band of 6-32 quarters."""
    G = gdp_growth()
    fig, axes = plt.subplots(1, 2, figsize=(9.33, 2.95))
    out = {}
    lo_nu, hi_nu = 1 / BC_BAND[1], 1 / BC_BAND[0]
    for ax, kind in zip(axes, ('growth', 'HP cycle')):
        ax.axvspan(lo_nu, hi_nu, color=st.Amber, alpha=0.15, lw=0, label='business-cycle band, 6-32 quarters' if kind == 'growth' else '_')
        for k, c in (('US', st.MainBlue), ('RO', st.IDAred)):
            x = G[k] if kind == 'growth' else hp_cycle(G[k + '_level'])
            nu, I = periodogram(x.values)
            fh = daniell(I, 2)
            ax.plot(nu, fh / np.mean((x - x.mean()) ** 2), color=c, lw=1.6,
                    label=('United States' if k == 'US' else 'Romania') if kind == 'growth' else '_')
            band = (nu >= lo_nu) & (nu <= hi_nu)
            j = int(np.argmax(fh))
            out[f'{k}.{kind}'] = {'share_bc': float(I[band].sum() / I.sum()), 'share_low': float(I[nu < lo_nu].sum() / I.sum()),
                                  'peak_period': float(1 / nu[j]), 'T': int(len(x))}
        ax.set_title(f'Spectrum of the {kind} / variance')
        ax.set_xlabel('frequency nu (cycles per quarter)')
        ax.set_xlim(0, 0.5)
    st.fig_legend_bottom(fig, ncol=3, y=0.03)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save('tsa_ch12_gdp_spectra', save_it)
    return out

## Run

In [ ]:
print(fig_gdp_series())
print(fig_gdp_spectra())

![tsa_ch12_gdp_series](./tsa_ch12_gdp_series.png)

Output: `tsa_ch12_gdp_series.pdf`

![tsa_ch12_gdp_spectra](./tsa_ch12_gdp_spectra.png)

Output: `tsa_ch12_gdp_spectra.pdf`